# Random Password Generator – Solution

**Extended Project** based on *Hassan S. Learn Python by Doing – Chapter 25*

Complete, runnable reference implementation with alternate approaches, extra practice solutions, and a Monte-Carlo simulation.

---


## Process Flowchart
![Flowchart](random_password_generator_flowchart.png)


## 1. Imports


In [ ]:
import random
import string
import secrets
import math
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np

print("Modules imported successfully.")


## 2. Character-pool helper


In [ ]:
AMBIGUOUS = set("0O1lI|`'\"")

def build_charset(use_upper=True, use_lower=True, use_digits=True, use_symbols=True, exclude_ambiguous=False):
    chars = ''
    if use_upper:
        chars += string.ascii_uppercase
    if use_lower:
        chars += string.ascii_lowercase
    if use_digits:
        chars += string.digits
    if use_symbols:
        chars += string.punctuation
    if exclude_ambiguous:
        chars = ''.join(c for c in chars if c not in AMBIGUOUS)
    return chars

# Demo
print("Full charset length:", len(build_charset()))
print("No-ambiguous length:", len(build_charset(exclude_ambiguous=True)))
print("Digits only:", build_charset(use_upper=False, use_lower=False, use_symbols=False))


## 3–4. Full `generate_password` (with ensure_each + secure)


In [ ]:
def generate_password(length=12, use_upper=True, use_lower=True, use_digits=True,
                      use_symbols=True, ensure_each=True, exclude_ambiguous=False,
                      min_length=6, secure=False):
    """Return (password, None) or (None, error_message)."""
    if length < min_length:
        return None, f"Password too short! Choose at least {min_length} characters."

    pools = {}
    if use_upper:
        p = string.ascii_uppercase
        if exclude_ambiguous:
            p = ''.join(c for c in p if c not in AMBIGUOUS)
        if p:
            pools['upper'] = p
    if use_lower:
        p = string.ascii_lowercase
        if exclude_ambiguous:
            p = ''.join(c for c in p if c not in AMBIGUOUS)
        if p:
            pools['lower'] = p
    if use_digits:
        p = string.digits
        if exclude_ambiguous:
            p = ''.join(c for c in p if c not in AMBIGUOUS)
        if p:
            pools['digits'] = p
    if use_symbols:
        p = string.punctuation
        if exclude_ambiguous:
            p = ''.join(c for c in p if c not in AMBIGUOUS)
        if p:
            pools['symbols'] = p

    if not pools:
        return None, "No character types selected (or all excluded as ambiguous)."

    all_chars = ''.join(pools.values())
    chooser = secrets.choice if secure else random.choice

    def multi_choice(seq, k):
        if secure:
            return [secrets.choice(seq) for _ in range(k)]
        return random.choices(seq, k=k)

    password_chars = []
    if ensure_each and length >= len(pools):
        for pool in pools.values():
            password_chars.append(chooser(pool))
        remaining = length - len(password_chars)
        password_chars.extend(multi_choice(all_chars, remaining))
    else:
        password_chars = multi_choice(all_chars, length)

    # Shuffle
    if secure:
        for i in range(len(password_chars) - 1, 0, -1):
            j = secrets.randbelow(i + 1)
            password_chars[i], password_chars[j] = password_chars[j], password_chars[i]
    else:
        random.shuffle(password_chars)

    return ''.join(password_chars), None

print("generate_password defined.")


## 5. Strength assessment


In [ ]:
def assess_strength(password):
    """Return (rating, score 0-10, details dict)."""
    if not password:
        return "Invalid", 0, {}
    length = len(password)
    has_upper = any(c.isupper() for c in password)
    has_lower = any(c.islower() for c in password)
    has_digit = any(c.isdigit() for c in password)
    has_symbol = any(c in string.punctuation for c in password)
    diversity = sum([has_upper, has_lower, has_digit, has_symbol])

    score = 0
    if length >= 16:
        score += 4
    elif length >= 12:
        score += 3
    elif length >= 10:
        score += 2
    elif length >= 8:
        score += 1
    score += diversity
    if diversity >= 3 and length >= 10:
        score += 1
    if diversity == 4 and length >= 12:
        score += 1
    score = min(score, 10)

    if score >= 8:
        rating = "Strong"
    elif score >= 5:
        rating = "Medium"
    else:
        rating = "Weak"

    # Book length safety net
    if length < 6:
        rating = "Weak"
    elif length <= 10 and rating == "Strong":
        rating = "Medium"

    details = {
        'length': length,
        'has_upper': has_upper,
        'has_lower': has_lower,
        'has_digit': has_digit,
        'has_symbol': has_symbol,
        'diversity': diversity,
        'score': score
    }
    return rating, score, details

print("assess_strength defined.")


## 6. Demo driver (reproducible)


In [ ]:
random.seed(42)

print("=== Basic demos (ensure_each=True) ===")
for L in [4, 8, 12, 16]:
    pw, err = generate_password(L, ensure_each=True)
    if err:
        print(f"len={L}: {err}")
    else:
        rating, score, det = assess_strength(pw)
        print(f"len={L}: {pw!r:20s}  [{rating:6s} score={score}]  diversity={det['diversity']}")

print("\n=== Cryptographically secure (secrets) ===")
pw, _ = generate_password(14, secure=True, ensure_each=True)
rating, score, _ = assess_strength(pw)
print(f"{pw!r}  →  {rating} (score={score})")

print("\n=== PIN-style (digits only) ===")
pw, _ = generate_password(6, use_upper=False, use_lower=False, use_symbols=False, ensure_each=False)
print(pw)

print("\n=== Exclude ambiguous characters ===")
pw, _ = generate_password(12, exclude_ambiguous=True, ensure_each=True)
print(pw)
print("Contains ambiguous?", any(c in AMBIGUOUS for c in pw))


## Alternate Implementation A – Pure `random.choice` loop + dictionary pools


In [ ]:
def generate_password_alt(length=12, use_upper=True, use_lower=True, use_digits=True,
                          use_symbols=True, ensure_each=True, exclude_ambiguous=False,
                          min_length=6):
    if length < min_length:
        return None, f"Password too short! Choose at least {min_length} characters."

    pools = {}
    mapping = [
        (use_upper, string.ascii_uppercase, 'upper'),
        (use_lower, string.ascii_lowercase, 'lower'),
        (use_digits, string.digits, 'digits'),
        (use_symbols, string.punctuation, 'symbols'),
    ]
    for flag, src, name in mapping:
        if flag:
            p = ''.join(c for c in src if c not in AMBIGUOUS) if exclude_ambiguous else src
            if p:
                pools[name] = p

    if not pools:
        return None, "No character types selected."

    all_chars = ''.join(pools.values())
    password_chars = []

    if ensure_each and length >= len(pools):
        for p in pools.values():
            password_chars.append(random.choice(p))
        for _ in range(length - len(password_chars)):
            password_chars.append(random.choice(all_chars))
    else:
        for _ in range(length):
            password_chars.append(random.choice(all_chars))

    random.shuffle(password_chars)
    return ''.join(password_chars), None

# Verify
random.seed(99)
pw, _ = generate_password_alt(12)
print("Alt result:", pw, assess_strength(pw)[0])


## Alternate Implementation B – List-comprehension + secrets only


In [ ]:
def generate_password_secrets_only(length=12, charset=None, min_length=6):
    """Minimal secure generator; caller supplies the charset string."""
    if length < min_length:
        return None, "Too short"
    if not charset:
        charset = string.ascii_letters + string.digits + string.punctuation
    # list-comp with secrets
    chars = [secrets.choice(charset) for _ in range(length)]
    # Fisher-Yates
    for i in range(len(chars)-1, 0, -1):
        j = secrets.randbelow(i+1)
        chars[i], chars[j] = chars[j], chars[i]
    return ''.join(chars), None

pw, _ = generate_password_secrets_only(16)
print("Secrets-only:", pw)
print("Strength:", assess_strength(pw)[0])


## More Practice – Solutions


In [ ]:
# 1. PIN mode
pin, _ = generate_password(6, use_upper=False, use_lower=False, use_symbols=False, ensure_each=False)
print("1. PIN:", pin)

# 2. High-security mode
hi, _ = generate_password(20, exclude_ambiguous=True, secure=True, ensure_each=True)
print("2. High-sec:", hi, assess_strength(hi)[0])

# 3. Batch generator
def generate_batch(n=5, **kwargs):
    seen = set()
    batch = []
    attempts = 0
    while len(batch) < n and attempts < n * 20:
        pw, err = generate_password(**kwargs)
        attempts += 1
        if err or pw in seen:
            continue
        seen.add(pw)
        batch.append(pw)
    return batch

print("3. Batch of 4 (len=10):", generate_batch(4, length=10))

# 4. Custom extra chars
def generate_with_extra(length=12, extra_chars="@#$", **kwargs):
    # simple approach: generate then inject, or enlarge pool
    base = build_charset(**{k: kwargs.get(k, True) for k in ['use_upper','use_lower','use_digits','use_symbols']},
                         exclude_ambiguous=kwargs.get('exclude_ambiguous', False))
    base += extra_chars
    # reuse secrets path
    return generate_password_secrets_only(length, charset=base)

print("4. With extra:", generate_with_extra(10)[0])

# 5. Strength histogram for length=10
random.seed(7)
ratings = Counter()
for _ in range(200):
    pw, _ = generate_password(10, ensure_each=True)
    ratings[assess_strength(pw)[0]] += 1
print("5. Histogram (len=10, n=200):", dict(ratings))


## Simulation Section (parameterised)
Change the values in the first cell below and re-execute to explore different regimes.


In [ ]:
# === SIMULATION PARAMETERS (edit these) ===
N_SAMPLES = 800
LENGTHS = list(range(6, 21))
ENSURE_EACH = True
EXCLUDE_AMBIGUOUS = False
USE_SYMBOLS = True
# ==========================================

random.seed(2026)
np.random.seed(2026)

strength_by_len = defaultdict(list)
scores_by_len = defaultdict(list)
entropy_est = defaultdict(list)

cs = build_charset(use_symbols=USE_SYMBOLS, exclude_ambiguous=EXCLUDE_AMBIGUOUS)
charset_size = len(cs)
print(f"Charset size used for entropy: {charset_size}")

per_len = max(20, N_SAMPLES // len(LENGTHS))
for L in LENGTHS:
    for _ in range(per_len):
        pw, err = generate_password(L, ensure_each=ENSURE_EACH,
                                    exclude_ambiguous=EXCLUDE_AMBIGUOUS,
                                    use_symbols=USE_SYMBOLS, min_length=4)
        if err:
            continue
        rating, score, _ = assess_strength(pw)
        strength_by_len[L].append(rating)
        scores_by_len[L].append(score)
        entropy_est[L].append(L * math.log2(charset_size))

avg_score = {L: np.mean(scores_by_len[L]) for L in LENGTHS if scores_by_len[L]}
pct_strong = {}
for L in LENGTHS:
    cnt = Counter(strength_by_len[L])
    total = sum(cnt.values()) or 1
    pct_strong[L] = 100.0 * cnt.get('Strong', 0) / total

print("Avg score (sample):", {k: round(v, 2) for k, v in list(avg_score.items())[::3]})
print("% Strong at L=8 / 12 / 16:",
      round(pct_strong.get(8, 0), 1),
      round(pct_strong.get(12, 0), 1),
      round(pct_strong.get(16, 0), 1))


In [ ]:
# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Random Password Generator – Simulation Results\n"
             f"(ensure_each={ENSURE_EACH}, symbols={USE_SYMBOLS}, exclude_amb={EXCLUDE_AMBIGUOUS})",
             fontsize=13, fontweight='bold')

# 1 Average score
ax = axes[0, 0]
ax.plot(list(avg_score.keys()), list(avg_score.values()), 'o-', color='#1565C0', lw=2, markersize=5)
ax.axhline(5, color='orange', ls='--', alpha=0.7, label='Medium ≥5')
ax.axhline(8, color='green', ls='--', alpha=0.7, label='Strong ≥8')
ax.set_xlabel("Password Length")
ax.set_ylabel("Average Strength Score (0-10)")
ax.set_title("Average Strength Score by Length")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 10.5)

# 2 % Strong
ax = axes[0, 1]
ax.bar(list(pct_strong.keys()), list(pct_strong.values()), color='#2E7D32', alpha=0.85, edgecolor='black')
ax.set_xlabel("Password Length")
ax.set_ylabel("% Rated Strong")
ax.set_title("Proportion of Strong Passwords")
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, 105)

# 3 Entropy
ax = axes[1, 0]
ents = [np.mean(entropy_est[L]) for L in LENGTHS]
ax.plot(LENGTHS, ents, 's-', color='#6A1B9A', lw=2)
ax.axhline(64, color='red', ls=':', label='~64-bit')
ax.axhline(128, color='darkred', ls=':', label='~128-bit')
ax.set_xlabel("Password Length")
ax.set_ylabel("Estimated Entropy (bits)")
ax.set_title("Charset Entropy ≈ length × log₂(|C|)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4 Overall score hist
ax = axes[1, 1]
all_scores = [s for L in LENGTHS for s in scores_by_len[L]]
ax.hist(all_scores, bins=range(0, 12), color='#0288D1', edgecolor='black', alpha=0.85, align='left')
ax.set_xlabel("Strength Score")
ax.set_ylabel("Frequency")
ax.set_title("Overall Score Distribution")
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("random_password_simulation.png", dpi=140, bbox_inches='tight', facecolor='white')
plt.show()
print("Chart saved → random_password_simulation.png")


## Key Observations from Simulation
- With `ensure_each=True` every password already has high diversity, so length becomes the dominant factor for the final rating.
- Crossing length 10–12 produces a sharp jump into the “Strong” category under the scoring rules used here.
- Entropy grows linearly with length; a 12-character password drawn from the full printable set already exceeds 70 bits.
- Turning symbols off or enabling `exclude_ambiguous` reduces charset size and therefore entropy – a useful trade-off discussion for usability vs security.


## Summary
You now have:
- A production-ready configurable generator (`generate_password`)
- A strength evaluator aligned with the book’s guidelines
- Two alternate coding styles
- Batch & special-mode helpers
- A fully parameterised Monte-Carlo simulation with visual analytics

Experiment with the simulation parameters to deepen your intuition about password strength.
